# AP3 Einarbeitung: SimPy an einer Kiste (Issue #17)

Lernmodell, **kein Ergebnis für den Verfahrensvergleich**. Es zeigt, wie SimPy funktioniert,
und benutzt dabei schon die echten Bausteine des Repos:

- Parameter aus `config/anlage.yaml` über `matsim.modell.anlage.wert`
- das Ereignisprotokoll von AP5 (`matsim.kennzahlen.ereignisprotokoll`)

Ablauf je Kiste (Charge): Kommissionierung → Hintransport → Lieferung an den Übergabeplatz →
Belegung der Station → Leerboxrückführung. Das entspricht den Methoden `kommissionierung`,
`hintransport`, `montage` und `leerboxrueckfuehrung` im Gerüst `src/matsim/modell/simulation.py`.

**Vereinfachungen:** kein Bedarfseingang und kein Planungsverfahren (die Chargen stehen fest),
kein Vorlauf vor dem ersten Pick, keine Quellboxen, Werkstücke stehen sofort an der Station bereit,
Fahrzeuge sind nicht verortet.

**Starten in VS Code:** einmal `pip install -e ".[dev,notebooks]"` in der `.venv`, oben rechts den
Kernel `.venv` wählen, dann Zelle für Zelle mit `Shift+Enter`.

In [ ]:
from dataclasses import asdict

import pandas as pd
import simpy

from matsim.kennzahlen.ereignisprotokoll import Ereignisprotokoll, Ereignistyp, Warteursache
from matsim.modell.anlage import lade_anlage, wert

anlage = lade_anlage()
STATIONEN = list(anlage["stationen"])

PICKZEIT_S = wert(anlage, "kommissionierplatz.pickzeit_je_teil_s")
KOMMISSIONIERPLAETZE = wert(anlage, "kommissionierplatz.gleichzeitige_chargen")
FAHRZEUGE = wert(anlage, "fahrzeuge.anzahl")
FAHRAUFTRAG_S = wert(anlage, "fahrzeuge.fahrauftrag_s")
LEERBOX_RUECK_S = wert(anlage, "fahrzeuge.leerboxruecktransport_s")
UEBERGABEPLAETZE = {s: wert(anlage, f"stationen.{s}.uebergabeplaetze") for s in STATIONEN}
BELEGUNG_S = {s: wert(anlage, f"stationen.{s}.belegung_je_werkstueck_s") for s in STATIONEN}
# Teile eines Satzes je Station, z. B. RASS1: [800, 814]
SATZ = {
    s: [pno for pno, e in anlage["stationen"][s]["materialsatz"].items() for _ in range(e["wert"])]
    for s in STATIONEN
}

print(f"Pickzeit je Teil {PICKZEIT_S} s, {FAHRZEUGE} Fahrzeuge, Fahrauftrag {FAHRAUFTRAG_S} s")
print(f"Übergabeplätze {UEBERGABEPLAETZE}")
print(f"Belegung je Werkstück {BELEGUNG_S}")
print(f"Satz je Station {SATZ}")

## Die vier Bausteine

1. **`simpy.Environment()`**: die Uhr. `env.now` ist die Simulationszeit in Sekunden. Sie springt
   von Ereignis zu Ereignis.
2. **Prozess**: eine Funktion mit `yield`. `yield env.timeout(23)` heißt „23 s beschäftigt“.
3. **Ressource**: alles Knappe. `anfrage = ressource.request()` stellt sich an, `yield anfrage`
   wartet, bis sie frei ist. Am Ende des `with`-Blocks (oder mit `release`) wird sie wieder frei.
4. **Ereignis**: alles, worauf man mit `yield` warten kann. Eine Anfrage ist auch ein Ereignis.
   `anfrage.triggered` ist schon beim Anstellen `True`, wenn die Ressource frei ist. Damit
   erkennen wir unten, ob überhaupt gewartet wird.

In [ ]:
def warte_auf(env, protokoll, anfrage, ursache, **felder):
    """Wartet auf eine Anfrage; protokolliert Beginn und Ende nur, wenn wirklich gewartet wird."""
    gewartet = not anfrage.triggered
    if gewartet:
        protokoll.schreibe(env.now, Ereignistyp.WARTEN_BEGINN, ursache=ursache, **felder)
    yield anfrage
    if gewartet:
        protokoll.schreibe(env.now, Ereignistyp.WARTEN_ENDE, ursache=ursache, **felder)


def charge(env, res, protokoll, charge_id, station, saetze, freigabe=None, geliefert=None):
    """Eine Kiste von der Kommissionierung bis zur Rückkehr der leeren Box."""
    if freigabe is not None:
        yield freigabe  # erst starten, wenn die vorherige Kiste geliefert ist
    ids = {"charge_id": charge_id, "box_id": f"Box_{charge_id}"}
    teile = SATZ[station] * saetze

    # 1) Kommissionierung: Kommissionierplatz belegen, je Teil ein Pick
    with res["kommissionierplatz"].request() as platz:
        yield from warte_auf(
            env, protokoll, platz, Warteursache.KOMMISSIONIERUNG, ressource="PICKSORT", **ids
        )
        protokoll.schreibe(
            env.now, Ereignistyp.KOMMISSIONIERUNG_BEGINN, ressource="PICKSORT", **ids
        )
        for pno in teile:
            yield env.timeout(PICKZEIT_S)
            protokoll.schreibe(
                env.now, Ereignistyp.TEIL_GEPICKT, ressource="PICKSORT", pno=pno, menge=1, **ids
            )
        protokoll.schreibe(env.now, Ereignistyp.KOMMISSIONIERUNG_ENDE, ressource="PICKSORT", **ids)

    # 2) Hintransport und Lieferung: Fahrzeug bleibt belegt, bis ein Übergabeplatz frei ist
    with res["fahrzeuge"].request() as fahrzeug:
        yield from warte_auf(
            env, protokoll, fahrzeug, Warteursache.TRANSPORT, ressource="fahrzeug", **ids
        )
        protokoll.schreibe(env.now, Ereignistyp.TRANSPORT_BEGINN, ressource="fahrzeug", **ids)
        yield env.timeout(FAHRAUFTRAG_S)
        protokoll.schreibe(env.now, Ereignistyp.TRANSPORT_ENDE, ressource=station, **ids)
        uebergabe = res["uebergabeplaetze"][station].request()  # ohne with: Freigabe erst in 4)
        yield from warte_auf(
            env, protokoll, uebergabe, Warteursache.STATIONSKAPAZITAET, ressource=station, **ids
        )
        protokoll.schreibe(env.now, Ereignistyp.LIEFERUNG, ressource=station, **ids)
    if geliefert is not None:
        geliefert.succeed()  # gibt die nächste Kiste frei (Experiment 3)

    # 3) Montage, vereinfacht: die Station ist je Satz so lange belegt wie ein Werkstück
    protokoll.schreibe(env.now, Ereignistyp.MONTAGE_BEGINN, ressource=station, **ids)
    yield env.timeout(saetze * BELEGUNG_S[station])
    protokoll.schreibe(env.now, Ereignistyp.MONTAGE_ENDE, ressource=station, **ids)

    # 4) Leerboxrückführung: Fahrzeug holen, Übergabeplatz freigeben, zurück zum Lager
    with res["fahrzeuge"].request() as fahrzeug:
        yield from warte_auf(
            env, protokoll, fahrzeug, Warteursache.TRANSPORT, ressource="fahrzeug", **ids
        )
        res["uebergabeplaetze"][station].release(uebergabe)
        protokoll.schreibe(env.now, Ereignistyp.LEERBOX_RUECK_BEGINN, ressource=station, **ids)
        yield env.timeout(LEERBOX_RUECK_S)
        protokoll.schreibe(env.now, Ereignistyp.LEERBOX_RUECK_ENDE, ressource="lager", **ids)


def laufe(chargen, fahrzeuge=FAHRZEUGE, uebergabeplaetze=None, nacheinander=False):
    """Baut das Modell auf, startet je Charge einen Prozess und lässt die Uhr laufen."""
    env = simpy.Environment()
    protokoll = Ereignisprotokoll()
    plaetze = (
        UEBERGABEPLAETZE if uebergabeplaetze is None else dict.fromkeys(STATIONEN, uebergabeplaetze)
    )
    res = {
        "kommissionierplatz": simpy.Resource(env, capacity=KOMMISSIONIERPLAETZE),
        "fahrzeuge": simpy.Resource(env, capacity=fahrzeuge),
        "uebergabeplaetze": {s: simpy.Resource(env, capacity=n) for s, n in plaetze.items()},
    }
    vorgaenger = None
    for charge_id, station, saetze in chargen:
        geliefert = env.event() if nacheinander else None
        env.process(charge(env, res, protokoll, charge_id, station, saetze, vorgaenger, geliefert))
        vorgaenger = geliefert
    env.run()  # läuft, bis keine Ereignisse mehr anstehen
    protokoll.schreibe(env.now, Ereignistyp.LAUF_ENDE)
    return protokoll


def mmss(sekunden):
    return f"{int(sekunden // 60):02d}:{int(sekunden % 60):02d}"


def als_tabelle(protokoll, picks=False):
    """Protokoll als Tabelle; einzelne Picks nur auf Wunsch."""
    df = pd.DataFrame([asdict(e) for e in protokoll.ereignisse])
    if not picks:
        df = df[df["ereignis"] != Ereignistyp.TEIL_GEPICKT]
    df = df.assign(zeit=df["zeit_s"].map(mmss), ereignis=df["ereignis"].astype(str))
    spalten = ["zeit", "ereignis", "charge_id", "ressource", "ursache", "pno"]
    return df[spalten].replace({0: "", "": ""}).reset_index(drop=True)


def kennzahlen(protokoll):
    """Laufende und Wartezeit je Ursache in Sekunden (Summe Ende minus Summe Beginn)."""
    df = pd.DataFrame([asdict(e) for e in protokoll.ereignisse])
    warten = df[df["ereignis"].isin([Ereignistyp.WARTEN_BEGINN, Ereignistyp.WARTEN_ENDE])]
    vorzeichen = warten["ereignis"].map({Ereignistyp.WARTEN_BEGINN: -1, Ereignistyp.WARTEN_ENDE: 1})
    je_ursache = (warten["zeit_s"] * vorzeichen).groupby(warten["ursache"]).sum()
    return {
        "ende": mmss(df["zeit_s"].max()),
        **{f"warten_{u}_s": int(t) for u, t in je_ursache.items()},
    }

## Basislauf: drei Kisten wie am 09.11.2026

Je Station eine Box mit drei Sätzen (9176/1, 9178/1, 9178/2), alle Kisten zum Zeitpunkt 0.

In [ ]:
CHARGEN_0911 = [("K9177", "RASS1", 3), ("K9179", "RASS2", 3), ("K9180", "RASS3", 3)]

basis = laufe(CHARGEN_0911)
print(kennzahlen(basis))
als_tabelle(basis)

## Experiment 1: Wie viele Fahrzeuge braucht es?

Ändere die Liste und führe die Zelle erneut aus. Wo verschiebt sich die Wartezeit hin?

In [ ]:
pd.DataFrame(
    {f"{n} Fahrzeug(e)": kennzahlen(laufe(CHARGEN_0911, fahrzeuge=n)) for n in [1, 2, 3]}
).fillna(0)

## Experiment 2: Ein Übergabeplatz, zwei Boxen für RASS3

Die zweite Box kommt an, während die erste noch an der Station steht. Das Fahrzeug wartet
beladen vor der Station (Ursache `stationskapazitaet`) und fehlt in der Zeit für andere Fahrten.

In [ ]:
voll = laufe([("K1", "RASS3", 8), ("K2", "RASS3", 8)], uebergabeplaetze=1)
print(kennzahlen(voll))
als_tabelle(voll)

## Experiment 3: Stationen nacheinander wie im echten Lauf

Am 09.11.2026 entstand der Kommissionierauftrag für die nächste Station erst, als die Box der
vorherigen abgeliefert war. `nacheinander=True` bildet das über ein eigenes Ereignis ab
(`env.event()` und `geliefert.succeed()` in `charge`).

Echt lagen zwischen dem Anlegen für RASS1 (15:05:15) und der Box an RASS3 (15:45:09) **39:54 min**.
Vergleiche das mit der Zeit der Lieferung an RASS3 im Modell. Woher kommt der Unterschied?

In [ ]:
nacheinander = laufe(CHARGEN_0911, nacheinander=True)
lieferungen = als_tabelle(nacheinander).query("ereignis == 'lieferung'")
lieferungen